# 🏗️ Construction Cost Engine - Complete Showcase

**ระบบคำนวณต้นทุนวัสดุก่อสร้างต่อหน่วย (Demo/Prototype)**

---

## 📋 Project Overview

- **Stack**: Next.js 16, TypeScript, React 19, Tailwind CSS 4
- **Materials**: 27 รายการวัสดุก่อสร้าง
- **Sources**: 11 แหล่งข้อมูล (Government: TPSO, CGD, DIT + Retail: 8 ห้าง)
- **Coverage**: 10 จังหวัดทั่วประเทศ
- **Features**: Price comparison, trend analysis, cost calculators

📂 **Repository**: `/construction-cost-engine`

## 🔧 Section 1: Setup

In [ ]:
!pip install pandas matplotlib seaborn tabulate -q
print("✅ Dependencies installed")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

from showcase_data import (
    MATERIALS, SOURCES, PROVINCES, BASE_PRICES, TRENDS, MONTH_LABELS,
    calc_wall_tile, calc_concrete
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Setup complete")
print(f"📦 {len(MATERIALS)} materials | {len(SOURCES)} sources | {len(PROVINCES)} provinces")

## 📦 Section 2: Data Overview

In [ ]:
# Materials DataFrame
df_mat = pd.DataFrame(MATERIALS).T
print("\n📊 MATERIALS (top 9)\n")
print(df_mat[['id', 'name', 'unit', 'cat']].head(9).to_string(index=False))

In [ ]:
# Category breakdown
print("\n📊 Materials by Category:\n")
for cat, count in df_mat['cat'].value_counts().items():
    print(f"  • {cat}: {count} items")

In [ ]:
# Data Sources
df_src = pd.DataFrame(SOURCES).T
print("\n📊 DATA SOURCES\n")
print(df_src[['name', 'type', 'mult']].to_string())

In [ ]:
# Provinces
df_prov = pd.DataFrame(PROVINCES)
print("\n📊 PROVINCES (10 จังหวัด)\n")
print(df_prov.to_string(index=False))

## 💰 Section 3: Price Data

In [ ]:
# Base prices (Bangkok, TPSO)
df_price = pd.DataFrame(list(BASE_PRICES.items()), columns=['Material', 'Price (THB)'])
df_price['Name'] = df_price['Material'].map(lambda x: MATERIALS.get(x, {}).get('name', ''))
df_price['Unit'] = df_price['Material'].map(lambda x: MATERIALS.get(x, {}).get('unit', ''))

print("\n💰 BASE PRICES (Bangkok, TPSO)\n")
print(df_price[['Material', 'Name', 'Price (THB)', 'Unit']].to_string(index=False))

## 📈 Section 4: Trend Analysis

In [ ]:
# Plot trends
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cement
axes[0].plot(MONTH_LABELS, TRENDS['CEMENT_001'], marker='o', color='#b94d2c', linewidth=2)
axes[0].set_title('ปูนซีเมนต์ Type I (บาท/ถุง 50กก)', fontsize=12, weight='bold')
axes[0].set_ylabel('Price (THB)')
axes[0].grid(alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Tile
axes[1].plot(MONTH_LABELS, TRENDS['TILE_001'], marker='s', color='#d97706', linewidth=2)
axes[1].set_title('กระเบื้อง 12x12" (บาท/ตร.ม.)', fontsize=12, weight='bold')
axes[1].set_ylabel('Price (THB)')
axes[1].grid(alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# Rebar
axes[2].plot(MONTH_LABELS, TRENDS['REBAR_DB12'], marker='^', color='#0e7c7b', linewidth=2)
axes[2].set_title('เหล็ก DB12 (บาท/ตัน)', fontsize=12, weight='bold')
axes[2].set_ylabel('Price (THB)')
axes[2].grid(alpha=0.3)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n📈 11-Month Price Trends (Aug 2568 - May 2569)")

## 🧮 Section 5: Calculator Demo A - Wall Tile

In [ ]:
# Example: 25.5 sqm wall tile job
area = 25.5
result = calc_wall_tile(
    area=area,
    tile_price=BASE_PRICES['TILE_001'],
    adhesive_price=BASE_PRICES['ADHESIVE_001'],
    grout_price=BASE_PRICES['GROUT_001']
)

print(f"\n🧮 {result['work_name']}")
print(f"   พื้นที่: {result['area']} ตร.ม.\n")

df_bom = pd.DataFrame(result['items'])
print(df_bom.to_string(index=False))

print(f"\n💰 Total Cost: {result['total']:,.2f} THB")
print(f"📊 Unit Cost: {result['unit_cost']:.2f} {result['unit_label']}")

## 🧮 Section 6: Calculator Demo B - Concrete

In [ ]:
# Example: 2.5 cu.m. concrete work
volume = 2.5
result = calc_concrete(
    volume=volume,
    cement_price=BASE_PRICES['CEMENT_001'],
    sand_price=BASE_PRICES['SAND_001'],
    rock_price=BASE_PRICES['ROCK_001']
)

print(f"\n🧮 {result['work_name']}")
print(f"   ปริมาตร: {result['volume']} ลบ.ม.\n")

df_bom = pd.DataFrame(result['items'])
print(df_bom.to_string(index=False))

print(f"\n💰 Total Cost: {result['total']:,.2f} THB")
print(f"📊 Unit Cost: {result['unit_cost']:.2f} {result['unit_label']}")

## 🎨 Section 7: Source Comparison Visualization

In [ ]:
# Compare CEMENT_001 across 6 sources
base = BASE_PRICES['CEMENT_001']
comparison = {
    'TPSO': base * 1.0,
    'CGD': base * 0.97,
    'HomePro': base * 1.08,
    'Global House': base * 0.95,
    'Thai Watsadu': base * 1.02,
    'BnB Home': base * 1.05
}

fig, ax = plt.subplots(figsize=(12, 6))
sources = list(comparison.keys())
prices = list(comparison.values())
colors = ['#1a3556', '#b94d2c', '#e30613', '#f37021', '#009a3d', '#003a70']

bars = ax.barh(sources, prices, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels
for i, (bar, price) in enumerate(zip(bars, prices)):
    ax.text(price + 2, i, f'{price:.2f}', va='center', fontsize=10, weight='bold')

ax.set_xlabel('Price (THB / ถุง 50กก)', fontsize=12)
ax.set_title('ราคาปูนซีเมนต์ Type I - เปรียบเทียบ 6 แหล่ง (Bangkok)', fontsize=14, weight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Price range: {:.2f} - {:.2f} THB".format(min(prices), max(prices)))
print(f"   Spread: {max(prices) - min(prices):.2f} THB ({(max(prices)/min(prices)-1)*100:.1f}%)")

## ✅ Section 8: Summary

In [ ]:
print("\n" + "="*60)
print(" 🏗️  CONSTRUCTION COST ENGINE - SHOWCASE COMPLETE")
print("="*60)
print(f"\n✅ Demonstrated:")
print(f"   • Data loading from {len(MATERIALS)} materials")
print(f"   • Price comparison across {len(SOURCES)} sources")
print(f"   • Historical trends (11 months)")
print(f"   • Cost calculators (Wall Tile, Concrete)")
print(f"   • Visualizations (trends, comparisons)")
print(f"\n📂 Project files:")
print(f"   • showcase.ipynb          - This notebook")
print(f"   • showcase_data.py        - Data module")
print(f"   • SHOWCASE.md             - Documentation")
print(f"\n🚀 To run the web app:")
print(f"   npm install && npm run dev")
print(f"   Open http://localhost:3000")
print("\n" + "="*60)